In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timedelta

# ==========================================================
# PARAMETERS
# ==========================================================

DB = "customer_health"
BRONZE = "BRONZE"
SILVER = "SILVER"

spark.conf.set("spark.sql.session.timeZone", "UTC")

try:
    transient_df = (
        raw_df
        .select(
            F.col("raw_data").alias("json_object"),
            "insert_date"
        )
    )
    
    transient_table = (
        f"{DB}.{SILVER}.LEAD_ACTIVITIES_PROCESSED_TRANSIENT"
    )
    
    transient_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(transient_table)

    # ==========================================================
    # READ BRONZE
    # ==========================================================

    raw_df = spark.table(
        f"{DB}.{BRONZE}.LEAD_ACTIVITES_RAW"
    )

    # ==========================================================
    # PARSE ACTIVITY ARRAY
    # ==========================================================

    parsed_df = (
        raw_df
        .withColumn(
            "activity_json_str",
            F.expr("customer_health.silver.parse_activity_final(raw_data)")
        )
        .withColumn(
            "activity_array",
            F.from_json(F.col("activity_json_str"), "array<string>")
        )
    )

    activities_df = (
        parsed_df
        .withColumn(
            "act",
            F.explode("activity_array")
        )
        .withColumn(
            "act",
            F.from_json(F.col("act"), "struct<_type:string,lead_id:string,contact_id:string,user_id:string,id:string,activity_at:string,date_created:string,date_updated:string,text:string,body_text:string,note:string,old_status_label:string,new_status_label:string,created_by_name:string,updated_by_name:string,attendees:string,custom_activity_type_id:string,starts_at:string,ends_at:string,status:string,created_by:string,updated_by:string,attachments:string,envelope:string,sender:string,opens:string,direction:string,subject:string,user_name:string,template_id:string,template_name:string,date_sent:string,has_reply:string>")
        )
    )

    # ==========================================================
    # EMAIL SOURCE
    # ==========================================================

    email_df = (
        activities_df
        .filter(
            F.col("act._type") == "Email"
        )
        .select(
            F.col("act.lead_id").alias("lead_id"),
            F.col("act.activity_at").cast("timestamp").alias("activity_at"),
            F.col("act.attachments").alias("attachments_json"),
            F.col("act.body_text").alias("body_text"),
            F.col("act.date_created").cast("timestamp").alias("date_created"),
            F.col("act.date_updated").cast("timestamp").alias("date_updated"),
            F.col("act.direction").alias("direction"),
            F.col("act.envelope").alias("envelope_json"),
            F.col("act.sender").alias("sender_json"),
            F.col("act.id").alias("activity_id"),
            F.col("act.opens").alias("opens_json"),
            F.col("act.sender").alias("sender"),
            F.col("act.status").alias("status"),
            F.col("act.subject").alias("subject"),
            F.col("act.user_id").alias("user_id"),
            F.col("act.user_name").alias("user_name"),
            F.col("act.template_id").alias("template_id"),
            F.col("act.template_name").alias("template_name"),
            F.col("act.date_sent").cast("timestamp").alias("date_sent"),
            F.col("act.has_reply").alias("has_reply"),
            F.col("act").alias("json_payload"),
            F.col("insert_date")
        )
    )

    email_window = Window.partitionBy(
        "activity_id",
        "activity_at",
        "date_sent"
    ).orderBy(
        F.col("insert_date").desc()
    )

    email_df = (
        email_df
        .withColumn(
            "rn",
            F.row_number().over(email_window)
        )
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    # ==========================================================
    # MERGE EMAIL TABLE
    # ==========================================================

    email_target = DeltaTable.forName(
        spark,
        f"{DB}.{SILVER}.LEAD_ACTIVITIES_EMAIL"
    )

    email_update_map = {
        "lead_id": "src.lead_id",
        "attachments_json": "src.attachments_json",
        "body_text": "src.body_text",
        "date_created": "src.date_created",
        "date_updated": "src.date_updated",
        "direction": "src.direction",
        "envelope_json": "src.envelope_json",
        "sender_json": "src.sender_json",
        "opens_json": "src.opens_json",
        "sender": "src.sender",
        "status": "src.status",
        "subject": "src.subject",
        "user_id": "src.user_id",
        "user_name": "src.user_name",
        "template_id": "src.template_id",
        "template_name": "src.template_name",
        "has_reply": "src.has_reply",
        "json_payload": "src.json_payload",
        "insert_date": "current_timestamp()"
    }

    email_insert_map = {
        "lead_id": "src.lead_id",
        "activity_at": "src.activity_at",
        "attachments_json": "src.attachments_json",
        "body_text": "src.body_text",
        "date_created": "src.date_created",
        "date_updated": "src.date_updated",
        "direction": "src.direction",
        "envelope_json": "src.envelope_json",
        "sender_json": "src.sender_json",
        "activity_id": "src.activity_id",
        "opens_json": "src.opens_json",
        "sender": "src.sender",
        "status": "src.status",
        "subject": "src.subject",
        "user_id": "src.user_id",
        "user_name": "src.user_name",
        "template_id": "src.template_id",
        "template_name": "src.template_name",
        "date_sent": "src.date_sent",
        "has_reply": "src.has_reply",
        "json_payload": "src.json_payload",
        "insert_date": "current_timestamp()"
    }

    (
        email_target.alias("tgt")
        .merge(
            email_df.alias("src"),
            """
            tgt.activity_id = src.activity_id
            AND tgt.activity_at = src.activity_at
            AND tgt.date_sent = src.date_sent
            """
        )
        .whenMatchedUpdate(set=email_update_map)
        .whenNotMatchedInsert(values=email_insert_map)
        .execute()
    )

    # ==========================================================
    # POPULATE TRANSIENT TABLE
    # ==========================================================

    (
        raw_df
        .select(
            F.col("raw_data").alias("json_object"),
            "insert_date"
        )
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"{DB}.{SILVER}.LEAD_ACTIVITIES_PROCESSED_TRANSIENT"
        )
    )

    transient_df = spark.table(
        f"{DB}.{SILVER}.LEAD_ACTIVITIES_PROCESSED_TRANSIENT"
    )

    # ==========================================================
    # DELETE EXISTING RECORDS
    # ==========================================================

    delete_keys = (
        transient_df
        .withColumn(
            "activity_json_str",
            F.expr("customer_health.silver.parse_activity_final(json_object)")
        )
        .withColumn(
            "activity_array",
            F.from_json(F.col("activity_json_str"), "array<string>")
        )
        .withColumn(
            "act_str",
            F.explode("activity_array")
        )
        .withColumn(
            "act",
            F.from_json(F.col("act_str"), "struct<lead_id:string,id:string>")
        )
        .select(
            F.col("act.lead_id").alias("lead_id"),
            F.col("act.id").alias("activity_id")
        )
        .distinct()
    )

    processed_target = DeltaTable.forName(
        spark,
        f"{DB}.{SILVER}.LEAD_ACTIVITIES_PROCESSED"
    )

    (
        processed_target.alias("tgt")
        .merge(
            delete_keys.alias("src"),
            """
            tgt.lead_id = src.lead_id
            AND tgt.activity_id = src.activity_id
            """
        )
        .whenMatchedDelete()
        .execute()
    )

    # ==========================================================
    # CRM SOURCE
    # ==========================================================

    crm_df = (
        transient_df
        .withColumn(
            "activity_json_str",
            F.expr("customer_health.silver.parse_activity_final(json_object)")
        )
        .withColumn(
            "activity_array",
            F.from_json(F.col("activity_json_str"), "array<string>")
        )
        .withColumn(
            "act_str",
            F.explode("activity_array")
        )
        .withColumn(
            "act",
            F.from_json(F.col("act_str"), "struct<_type:string,lead_id:string,contact_id:string,user_id:string,id:string,activity_at:string,date_created:string,date_updated:string,text:string,body_text:string,note:string,old_status_label:string,new_status_label:string,created_by_name:string,updated_by_name:string,attendees:string,custom_activity_type_id:string,starts_at:string,ends_at:string,status:string,created_by:string,updated_by:string>")
        )
        .select(
            F.col("act.id").alias("activity_id"),
            F.col("act.lead_id").alias("lead_id"),
            F.col("act.contact_id").alias("contact_id"),
            F.col("act.user_id").alias("user_id"),
            F.col("act._type").alias("type"),

            F.when(
                F.col("act._type") == "SMS",
                F.col("act.text")
            )
            .when(
                F.col("act._type") == "Email",
                F.col("act.body_text")
            )
            .when(
                F.col("act._type") == "Note",
                F.col("act.note")
            )
            .alias("text"),

            F.concat(
                F.coalesce(F.col("act.old_status_label"), F.lit("")),
                F.lit(" -> "),
                F.coalesce(F.col("act.new_status_label"), F.lit(""))
            ).alias("status_change"),

            F.col("act.created_by_name").alias("created_by_name"),
            F.col("act.updated_by_name").alias("updated_by_name"),
            F.col("act.attendees").alias("attendees"),

            F.col("act.custom_activity_type_id").alias("custom_activity_id"),

            F.col("act.starts_at").cast("timestamp").alias("meeting_starts_at"),
            F.col("act.ends_at").cast("timestamp").alias("meeting_end_at"),

            F.col("act.status").alias("meeting_status"),

            F.col("act.activity_at").cast("timestamp").alias("activity_at"),
            F.col("act.date_created").cast("timestamp").alias("date_created"),
            F.col("act.date_updated").cast("timestamp").alias("date_updated"),

            F.col("act.created_by").alias("created_by_user"),
            F.col("act.updated_by").alias("updated_by_user")
        )
    )

    crm_df = (
        crm_df
        .withColumn(
            "insert_date",
            F.current_timestamp()
        )
        .withColumn(
            "update_date",
            F.current_timestamp()
        )
        .withColumn(
            "md5_hash",
            F.md5(
                F.concat_ws(
                    "",
                    F.coalesce(F.col("contact_id").cast("string"), F.lit("")),
                    F.coalesce(F.col("type").cast("string"), F.lit("")),
                    F.coalesce(F.col("text").cast("string"), F.lit("")),
                    F.coalesce(F.col("status_change").cast("string"), F.lit("")),
                    F.coalesce(F.col("created_by_name").cast("string"), F.lit("")),
                    F.coalesce(F.col("updated_by_name").cast("string"), F.lit("")),
                    F.coalesce(F.col("attendees").cast("string"), F.lit("")),
                    F.coalesce(F.col("meeting_starts_at").cast("string"), F.lit("")),
                    F.coalesce(F.col("meeting_end_at").cast("string"), F.lit("")),
                    F.coalesce(F.col("meeting_status").cast("string"), F.lit("")),
                    F.coalesce(F.col("created_by_user").cast("string"), F.lit("")),
                    F.coalesce(F.col("updated_by_user").cast("string"), F.lit(""))
                )
            )
        )
    )

    # ==========================================================
    # MERGE PROCESSED TABLE
    # ==========================================================

    update_map = {
        "lead_id": "src.lead_id",
        "contact_id": "src.contact_id",
        "user_id": "src.user_id",
        "type": "src.type",
        "text": "src.text",
        "status_change": "src.status_change",
        "created_by_name": "src.created_by_name",
        "updated_by_name": "src.updated_by_name",
        "attendees": "src.attendees",
        "custom_activity_id": "src.custom_activity_id",
        "meeting_starts_at": "src.meeting_starts_at",
        "meeting_end_at": "src.meeting_end_at",
        "meeting_status": "src.meeting_status",
        "activity_at": "src.activity_at",
        "date_created": "src.date_created",
        "date_updated": "src.date_updated",
        "created_by_user": "src.created_by_user",
        "updated_by_user": "src.updated_by_user",
        "update_date": "current_timestamp()",
        "md5_hash": "src.md5_hash"
    }

    insert_map = {
        "activity_id": "src.activity_id",
        "lead_id": "src.lead_id",
        "contact_id": "src.contact_id",
        "user_id": "src.user_id",
        "type": "src.type",
        "text": "src.text",
        "status_change": "src.status_change",
        "created_by_name": "src.created_by_name",
        "updated_by_name": "src.updated_by_name",
        "attendees": "src.attendees",
        "custom_activity_id": "src.custom_activity_id",
        "meeting_starts_at": "src.meeting_starts_at",
        "meeting_end_at": "src.meeting_end_at",
        "meeting_status": "src.meeting_status",
        "activity_at": "src.activity_at",
        "date_created": "src.date_created",
        "date_updated": "src.date_updated",
        "created_by_user": "src.created_by_user",
        "updated_by_user": "src.updated_by_user",
        "insert_date": "current_timestamp()",
        "update_date": "current_timestamp()",
        "md5_hash": "src.md5_hash"
    }

    (
        processed_target.alias("tgt")
        .merge(
            crm_df.alias("src"),
            "tgt.activity_id = src.activity_id"
        )
        .whenMatchedUpdate(
            condition="tgt.md5_hash <> src.md5_hash",
            set=update_map
        )
        .whenNotMatchedInsert(
            values=insert_map
        )
        .execute()
    )


    print("Store Procedure Executed Successfully")

except Exception as e:
    raise Exception(
        f"Error: {str(e)}"
    )